<a href="https://colab.research.google.com/github/awaiskhan005/DATA-SCIENCE-AND-AI-/blob/main/Flight_Radar_Project_Data_Scrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fr24sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.8 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.2
    Uninstalling tenacity-9.1.2:
      Successfully uninstalled tenacity-9.1.2
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pydantic_core-2.33.2:
      Successfully uninstalled pydantic_core-2.33.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.7
    Uninstalling pydantic-2.11.7:
      Successfully uninstalled pydantic-2.11.7
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. 

In [2]:
import os

In [4]:
# Windows (PowerShell)
# The syntax `osEnv:VARIABLE_NAME="value"` is not valid Python.
# Use os.environ to set environment variables in Python.
os.environ['FR24_API_TOKEN'] = "0198c1ec-2691-702b-938b-648c76010311|h67I1sZyX3kfpPNjWdy9ZdqNCZ7zQXDWXE8kL1hK61bf21b1"

In [5]:
from fr24sdk.client import Client

# Option A: token from FR24_API_TOKEN env var
client = Client()

# Option B: pass token explicitly
client = Client(api_token="0198c1ec-2691-702b-938b-648c76010311|h67I1sZyX3kfpPNjWdy9ZdqNCZ7zQXDWXE8kL1hK61bf21b1")

In [6]:
from fr24sdk.client import Client

with Client() as client:
    waw = client.airports.get_full("WAW")
    print(waw.name, waw.icao, waw.city, waw.country, waw.lat, waw.lon)

Warsaw Chopin Airport EPWA Warsaw code='PL' name='POLAND' 52.165749 20.967119


In [9]:
from fr24sdk.client import Client

bounds = "39.8,24.5,44.0,63.4"  # N, S, W, E
with Client() as client:
    flights = client.live.flight_positions.get_light(bounds=bounds)
    print(f"Flights returned: {len(flights.data)}")

Flights returned: 300


In [10]:
from fr24sdk.client import Client

with Client() as client:
    summary = client.flight_summary.get_light(flight_ids=["391fdd79"])  # example flight_id
    tracks = client.flight_tracks.get(flight_id=["391fdd79"])
    print(summary, tracks)

data=[FlightSummaryLight(fr24_id='391fdd79', flight='D84529', callsign='NSZ4529', operating_as='NSZ', painted_as='NSZ', type='B38M', reg='SE-RTC', orig_icao='ESSA', datetime_takeoff=None, dest_icao='GMAD', dest_icao_actual='GMAD', datetime_landed=None, hex='4ACA83', first_seen='2025-02-14T11:47:06Z', last_seen='2025-02-14T13:11:49Z', flight_ended=True)] data=[FlightTracks(fr24_id='391fdd79', tracks=[FlightTrackPoint(timestamp='2025-02-14T11:47:06Z', lat=59.65126, lon=17.92569, alt=0, gspeed=0, vspeed=0, track=129, squawk='2000', callsign='NSZ4529', source='ADSB')])]


In [11]:
from fr24sdk.client import Client

# Approximate bounds for Iranian airspace (N, S, W, E)
iran_bounds = "39.8,24.5,44.0,63.4"

with Client() as client:
    # Get historical flights within the specified bounds
    # Note: The historical API might require specific parameters or a paid subscription
    # to get data based on bounds and timeframes.
    # This is a basic example. You might need to adjust parameters based on API documentation.
    # You might need to specify a time range as well.
    try:
        historical_flights = client.live.flight_positions.get_historical(bounds=iran_bounds)
        print(f"Historical flights returned: {len(historical_flights.data)}")
        # display(historical_flights.data) # Uncomment to see the data
    except Exception as e:
        print(f"Error fetching historical data: {e}")


    # Get live flights within the specified bounds
    try:
        live_flights = client.live.flight_positions.get_light(bounds=iran_bounds)
        print(f"Live flights returned: {len(live_flights.data)}")
        # display(live_flights.data) # Uncomment to see the data
    except Exception as e:
        print(f"Error fetching live data: {e}")

Error fetching historical data: 'LivePositionsResource' object has no attribute 'get_historical'
Live flights returned: 300


In [12]:
import requests
import datetime
import folium

def get_flights_for_airport_date(route: str, date_str: str, headers):
    """
    Fetch all flights on a specific route on the specified date,
    using Flight Summary Light endpoint.
    """
    base_url = "https://fr24api.flightradar24.com/api/flight-summary/light"

    # Build the start/end of day in UTC for the query (example uses the entire 24-hour period)
    flight_datetime_from = f"{date_str} 00:00:00"
    flight_datetime_to = f"{date_str} 23:59:59"

    params = {
        "flight_datetime_from": flight_datetime_from,
        "flight_datetime_to": flight_datetime_to,
        "routes": f"{route}",
        "limit": 50  # Adjust limit as needed.
    }

    response = requests.get(base_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()

    # Extract flight IDs from the response to then fetch detailed track data.
    fr24_ids = []

    for flight in data.get("data", []):
        fr24_id = flight.get("fr24_id")
        if fr24_id:
            fr24_ids.append(fr24_id)

    return fr24_ids

def get_flight_tracks(fr24_ids, headers):
    """
    Fetch full flight trajectories using the FR24 Flight Tracks endpoint.
    """

    base_url = "https://fr24api.flightradar24.com/api/flight-tracks"

    all_tracks = []

    for flight_id in fr24_ids:
        params = {
            "flight_id": flight_id
        }
        response = requests.get(base_url, params=params, headers=headers)
        response.raise_for_status()
        track_data = response.json()
        # Store track info in a list (or write to DB, etc.)
        all_tracks.append(track_data)

    return all_tracks

def plot_flight_tracks_on_map(tracks_data, output_html="all_flight_tracks.html"):
    """
    Plots all flight tracks on a folium map and saves to an HTML file.
    """
    # Center map somewhere over Europe by default (adjust as needed)
    fmap = folium.Map(location=[59.0, 18.0], zoom_start=5)

    for track in tracks_data:
        if not track:
            continue

        positions = track[0].get("tracks", [])

        # Extract (lat, lon) for each position to form a list of coordinates
        coords = []
        for pos in positions:
            lat = pos.get("lat")
            lon = pos.get("lon")

            # Only add valid points
            if lat is not None and lon is not None:
                coords.append((lat, lon))

        # Add a polyline for this flight
        if coords:
            folium.PolyLine(coords, color="blue", weight=2.5, opacity=1).add_to(fmap)

    # Save the result to an HTML file
    fmap.save(output_html)
    print(f"Map with all flight tracks saved to: {output_html}")

def main():
    API_KEY = "0198c1ec-2691-702b-938b-648c76010311|h67I1sZyX3kfpPNjWdy9ZdqNCZ7zQXDWXE8kL1hK61bf21b1"  # Replace with your actual API key

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json",
        "Accept-Version": "v1"
    }

    # Example: All flights departing from LHR to JFK on 2025-01-05
    origin = "LHR"
    destination = "JFK"
    date_str = "2025-01-05"

    route = f"{origin}-{destination}"

    print(f"Fetching flights for route {route} on date {date_str} ...")
    flight_ids = get_flights_for_airport_date(route, date_str, headers)
    print(f"Found {len(flight_ids)} flights. Fetching tracks now...")

    flight_tracks = get_flight_tracks(flight_ids, headers)
    print(f"Fetched tracks for {len(flight_tracks)} flights.")

    plot_flight_tracks_on_map(flight_tracks, output_html="all_flight_tracks.html")

if __name__ == "__main__":
    main()

Fetching flights for route LHR-JFK on date 2025-01-05 ...
Found 18 flights. Fetching tracks now...
Fetched tracks for 18 flights.
Map with all flight tracks saved to: all_flight_tracks.html
